## Quantum Single-Qubit Update With Upper/Lower Indices

For a target qubit $q$ in an $n$-qubit statevector, split the flat basis index into:

- lower part: $idx_{lower} \in [0, 2^q)$
- upper part: $idx_{upper} = u \cdot 2^{q+1}$ with $u \in [0, 2^{n-q-1})$

Then compute paired amplitudes:

- $idx0 = idx_{lower} + idx_{upper} + 0 \cdot 2^q$
- $idx1 = idx_{lower} + idx_{upper} + 1 \cdot 2^q$

and apply the 2x2 gate matrix to $(\psi[idx0], \psi[idx1])$.

In [1]:
import numpy as np


def _prepare_apply_u_inputs(
    state_vec: np.ndarray, gate_2x2: np.ndarray, q: int, n_qubits: int
) -> tuple[np.ndarray, np.ndarray, int, int]:
    """Shared validation and precomputation for single-qubit apply_u functions."""
    flat_in = np.asarray(state_vec, dtype=np.complex128).reshape(-1)
    gate = np.asarray(gate_2x2, dtype=np.complex128)

    if flat_in.size != (1 << n_qubits):
        raise ValueError("State length must be 2**n_qubits")
    if gate.shape != (2, 2):
        raise ValueError("gate_2x2 must have shape (2, 2)")
    if q < 0 or q >= n_qubits:
        raise ValueError("q must satisfy 0 <= q < n_qubits")

    lower = 2**q
    upper = 2 ** (n_qubits - q - 1)
    return flat_in, gate, lower, upper


def _extract_u_entries(gate: np.ndarray) -> tuple[complex, complex, complex, complex]:
    """Return gate matrix entries in row-major order."""
    return gate[0, 0], gate[0, 1], gate[1, 0], gate[1, 1]


def _apply_u_loop_body(
    flat_in: np.ndarray,
    flat_out: np.ndarray,
    lower: int,
    upper: int,
    q: int,
    u00: complex,
    u01: complex,
    u10: complex,
    u11: complex,
) -> None:
    """Shared core loop for applying a single-qubit gate on a flat statevector."""
    for upper_idx in range(upper):
        idx_upper = upper_idx * (2 ** (q + 1))
        for idx_lower in range(lower):
            idx0 = idx_lower + idx_upper
            idx1 = idx_lower + idx_upper + 2**q

            a0 = flat_in[idx0]
            a1 = flat_in[idx1]
            flat_out[idx0] = u00 * a0 + u01 * a1
            flat_out[idx1] = u10 * a0 + u11 * a1


def apply_u_loop_upper_lower(state_vec: np.ndarray, gate_2x2: np.ndarray, q: int, n_qubits: int) -> np.ndarray:
    """Apply a single-qubit gate via explicit upper/lower index loops."""
    flat_in, gate, lower, upper = _prepare_apply_u_inputs(state_vec, gate_2x2, q, n_qubits)
    flat_out = np.empty_like(flat_in)
    u00, u01, u10, u11 = _extract_u_entries(gate)

    _apply_u_loop_body(flat_in, flat_out, lower, upper, q, u00, u01, u10, u11)
    return flat_out


def apply_u_reference_tensor(state_vec: np.ndarray, gate_2x2: np.ndarray, q: int, n_qubits: int) -> np.ndarray:
    """Reference update using tensor contraction for verification."""
    state_t = np.asarray(state_vec, dtype=np.complex128).reshape([2] * n_qubits)
    gate_t = np.asarray(gate_2x2, dtype=np.complex128).reshape(2, 2)

    # State tensor axes are [q_{n-1}, ..., q_0].
    axis_in = q
    axis_out = n_qubits

    state_axes = list(range(n_qubits))[::-1]
    out_axes = state_axes.copy()
    out_axes[n_qubits - 1 - axis_in] = axis_out

    updated = np.einsum(
        gate_t,
        [axis_out, axis_in],
        state_t,
        state_axes,
        out_axes,
        optimize=False,
    )
    return updated.reshape(-1)


# Demo / correctness check
rng = np.random.default_rng(7)
n = 5
q = 2
psi = rng.normal(size=(1 << n)) + 1j * rng.normal(size=(1 << n))
psi = psi / np.linalg.norm(psi)

# Random unitary from QR
X = rng.normal(size=(2, 2)) + 1j * rng.normal(size=(2, 2))
Q, _ = np.linalg.qr(X)
U = Q

psi_loop = apply_u_loop_upper_lower(psi, U, q=q, n_qubits=n)
psi_ref = apply_u_reference_tensor(psi, U, q=q, n_qubits=n)
# print('psi_loop:', psi_loop)
# print('psi_ref:', psi_ref)

print("Loop vs reference close?", np.allclose(psi_loop, psi_ref, atol=1e-16))

Loop vs reference close? True


In [2]:
import numba as nb
from time import perf_counter
import numpy as np

# Reuse the exact same loop body as the Python implementation, then JIT it.
_apply_u_numba_kernel = nb.njit(cache=True)(_apply_u_loop_body)


def apply_u_numba(state_vec: np.ndarray, gate_2x2: np.ndarray, q: int, n_qubits: int) -> np.ndarray:
    """Apply a single-qubit gate using shared prep plus a Numba-compiled shared loop body."""
    flat_in, gate, lower, upper = _prepare_apply_u_inputs(state_vec, gate_2x2, q, n_qubits)
    flat_out = np.empty_like(flat_in)
    u00, u01, u10, u11 = _extract_u_entries(gate)

    _apply_u_numba_kernel(flat_in, flat_out, lower, upper, q, u00, u01, u10, u11)
    return flat_out


def apply_u_einsum(state_vec: np.ndarray, gate_2x2: np.ndarray, q: int, n_qubits: int) -> np.ndarray:
    """Apply a single-qubit gate using an einsum contraction."""
    state_t = np.asarray(state_vec, dtype=np.complex128).reshape([2] * n_qubits)
    gate_t = np.asarray(gate_2x2, dtype=np.complex128).reshape(2, 2)

    axis_in = q
    axis_out = n_qubits

    state_axes = list(range(n_qubits))[::-1]
    out_axes = state_axes.copy()
    out_axes[n_qubits - 1 - axis_in] = axis_out

    updated = np.einsum(
        gate_t,
        [axis_out, axis_in],
        state_t,
        state_axes,
        out_axes,
        optimize=False,
    )
    return updated.reshape(-1)

In [3]:
def benchmark_apply_u(fn, psi: np.ndarray, U: np.ndarray, q: int, n: int, repeats: int = 50) -> float:
    times = np.empty(repeats, dtype=np.float64)
    for r in range(repeats):
        t0 = perf_counter()
        fn(psi, U, q, n)
        t1 = perf_counter()
        times[r] = (t1 - t0) * 1000.0
    return float(np.median(times))


# Benchmark setup
rng = np.random.default_rng(123)
n = 16  # number of qubits
q = 5  # acting on
psi = rng.normal(size=(1 << n)) + 1j * rng.normal(size=(1 << n))
psi = psi / np.linalg.norm(psi)
X = rng.normal(size=(2, 2)) + 1j * rng.normal(size=(2, 2))
Q, _ = np.linalg.qr(X)
U = Q

# Warm-up for JIT compilation
_ = apply_u_numba(psi, U, q, n)

psi_numba = apply_u_numba(psi, U, q, n)
psi_loop = apply_u_loop_upper_lower(psi, U, q, n)
psi_einsum = apply_u_einsum(psi, U, q, n)

print("numba vs loop close?", np.allclose(psi_numba, psi_loop, atol=1e-12))
print("numba vs einsum close?", np.allclose(psi_numba, psi_einsum, atol=1e-12))

ms_numba = benchmark_apply_u(apply_u_numba, psi, U, q, n, repeats=40)
ms_loop = benchmark_apply_u(apply_u_loop_upper_lower, psi, U, q, n, repeats=40)
ms_einsum = benchmark_apply_u(apply_u_einsum, psi, U, q, n, repeats=40)

print()
print("Median runtime over 40 runs [ms]")
print(f"apply_u_numba: {ms_numba:9.3f}")
print(f"apply_u_loop:  {ms_loop:9.3f}")
print(f"apply_u_einsum:{ms_einsum:9.3f}")
print()
print(f"speedup numba vs loop:   {ms_loop / ms_numba:6.2f}x")
print(f"speedup numba vs einsum: {ms_einsum / ms_numba:6.2f}x")

numba vs loop close? True
numba vs einsum close? True

Median runtime over 40 runs [ms]
apply_u_numba:     0.343
apply_u_loop:     18.187
apply_u_einsum:    0.636

speedup numba vs loop:    53.00x
speedup numba vs einsum:   1.85x


In [4]:
import numpy as np


def einsum_matmul(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """Reference using NumPy einsum: 'ij,jk->ik'."""
    return np.einsum("ij,jk->ik", A, B)


def loop_matmul_from_vectors(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """
    Manual implementation of 'ij,jk->ik' using for-loops after
    flattening both operands into 1D vectors.
    """
    A = np.asarray(A)
    B = np.asarray(B)

    if A.ndim != 2 or B.ndim != 2:
        raise ValueError("A and B must both be 2D arrays")

    m, n = A.shape
    n2, p = B.shape
    if n != n2:
        raise ValueError("Inner dimensions must match")

    # Convert both matrices to vectors first
    A_vec = A.reshape(-1)
    B_vec = B.reshape(-1)

    # Output matrix
    C = np.zeros((m, p), dtype=np.result_type(A.dtype, B.dtype))

    # C[i, k] = sum_j A[i, j] * B[j, k]
    # A[i, j] in A_vec is at i*n + j
    # B[j, k] in B_vec is at j*p + k
    for i in range(m):
        for k in range(p):
            total = 0.0
            for j in range(n):
                total += A_vec[i * n + j] * B_vec[j * p + k]
            C[i, k] = total

    return C

In [5]:
# Demo data
rng = np.random.default_rng(42)
A = rng.normal(size=(3, 4))
B = rng.normal(size=(4, 2))

C_einsum = einsum_matmul(A, B)
C_loops = loop_matmul_from_vectors(A, B)

print("einsum result:", C_einsum)
print("loop result:", C_loops)
print("close?", np.allclose(C_einsum, C_loops, atol=1e-10))

einsum result: [[ 0.63688722  0.47058701]
 [-0.96827145 -1.18712946]
 [ 0.60761486 -0.16799613]]
loop result: [[ 0.63688722  0.47058701]
 [-0.96827145 -1.18712946]
 [ 0.60761486 -0.16799613]]
close? True
